# Project 17: Supplier Quality Scorecard, analisis Q1 sampai Q14
Notebook ini membaca CSV di `results/2-analysis` dan `results/1-data-preparation`. Data adalah simulasi yang dianonimkan.

In [1]:
from pathlib import Path
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
R2 = Path('../2-analysis'); R1 = Path('../1-data-preparation')
pd.set_option('display.width', 160)

## Q1 dan Q2: NG rate per supplier dan uji antar supplier

In [2]:
q1 = pd.read_csv(R2/'q1_ng_rate_ci.csv'); display(q1[q1.tingkat=='supplier'][['supplier_code','units','ng','ng_rate','ci_low','ci_high']])
display(pd.read_csv(R2/'q1_top_per_material.csv')); display(pd.read_csv(R2/'q2_tests.csv'))

,supplier_code,units,ng,ng_rate,ci_low,ci_high
0,SUP-A,111746,2195,0.019643,0.018845,0.020473
1,SUP-B,17712,229,0.012929,0.011368,0.014702
2,SUP-C,118113,4167,0.035280,0.034243,0.036347
3,SUP-D,154970,1947,0.012564,0.012021,0.013130
4,SUP-E,157113,1856,0.011813,0.011291,0.012359


,material_code,supplier_tertinggi,ng_rate,ci_low,ci_high,pembanding,p_unit_holm_maks,p_lot_holm_maks,supplier_kedua,ng_rate_kedua
0,MAT-1,SUP-C,0.037732,0.036177,0.039351,1,0.000000e+00,4.997167e-40,SUP-D,0.012564
1,MAT-2,SUP-C,0.033114,0.031742,0.034544,1,0.000000e+00,1.845723e-18,SUP-A,0.019271
2,MAT-3,SUP-A,0.020022,0.018888,0.021223,2,4.408871e-10,1.162531e-02,SUP-B,0.012929


,uji,statistik,derajat_bebas,p_value,effect_size_cramers_v,n_unit
0,chi-square NG vs OK antar 5 supplier (unit),2547.551264,4,0.000000e+00,0.067469,559654
1,chi-square antar supplier pada MAT-1 (unit),1371.890221,1,2.696422e-300,0.080756,210361
2,chi-square antar supplier pada MAT-2 (unit),219.611778,1,1.099170e-49,0.042936,119129
3,chi-square antar supplier pada MAT-3 (unit),202.833514,2,9.021153e-45,0.029686,230164
4,Kruskal-Wallis NG rate lot antar 5 supplier (lot),403.407697,4,5.104976e-86,NaN,1164


## Q3: ranking bulanan

In [3]:
nm = pd.read_csv(R2/'ng_month.csv'); display(nm.pivot(index='month', columns='supplier_code', values='rank_ng')); display(pd.read_csv(R2/'q3_rank_summary.csv'))

supplier_code,SUP-A,SUP-B,SUP-C,SUP-D,SUP-E
month,,,,,
2026-03,3,2,1,4,5
2026-04,4,5,1,2,3
2026-05,2,3,1,5,4
2026-06,4,2,1,3,5
2026-07,2,5,1,3,4
2026-08,2,4,1,5,3


,supplier_code,rank_rata_rata,rank_terbaik,rank_terburuk,bulan_rank_1,perubahan_rank,jumlah_bulan
0,SUP-A,2.833333,2,4,0,4,6
1,SUP-B,3.500000,2,5,0,5,6
2,SUP-C,1.000000,1,1,6,0,6
3,SUP-D,3.666667,2,5,0,4,6
4,SUP-E,4.000000,3,5,0,5,6


## Q4 dan Q5: variasi antar lot dan lot di atas ambang

In [4]:
display(pd.read_csv(R2/'q4_lot_variation.csv')); display(pd.read_csv(R2/'q5_lot_threshold.csv').query('dasar==1'))

,supplier_code,lots_eval,mean_lot_ng_rate,sd_lot_ng_rate,lot_cv,lot_di_atas_ucl,persen_lot_di_atas_ucl,dispersi_berlebih
0,SUP-A,228,0.020727,0.025834,1.246389,24,0.105263,15.351318
1,SUP-B,40,0.013051,0.011633,0.891375,2,0.050000,4.147196
2,SUP-C,243,0.035408,0.034896,0.985554,18,0.074074,16.732753
3,SUP-D,333,0.012343,0.012940,1.048356,24,0.072072,5.971055
4,SUP-E,320,0.011524,0.013064,1.133594,22,0.068750,6.863906


,ambang,supplier_code,lot_dinilai,lot_di_atas_ambang,persen,ci_low,ci_high,dasar
5,0.02,SUP-A,228,63,0.276316,0.222342,0.337703,1
6,0.02,SUP-B,40,4,0.100000,0.039580,0.230518,1
7,0.02,SUP-C,243,205,0.843621,0.792649,0.883899,1
8,0.02,SUP-D,333,27,0.081081,0.056320,0.115397,1
9,0.02,SUP-E,320,26,0.081250,0.056048,0.116386,1


## Q6 dan Q7: rolling 4 minggu dan perubahan antar bulan

In [5]:
w = pd.read_csv(R2/'rolling_weekly.csv', parse_dates=['week_start'])
fig, ax = plt.subplots(figsize=(8,3))
for s, g in w.groupby('supplier_code'): ax.plot(g.week_start, g.rolling_ng_rate*100, label=s)
ax.legend(); ax.set_ylabel('NG rate rolling 4 minggu (%)'); plt.show()
display(pd.read_csv(R2/'q6_worsening.csv')); display(pd.read_csv(R2/'q7_month_change.csv').query('nyata==1'))

,supplier_code,ng_rate_terkini,ng_rate_sebelumnya,rasio,z,p_satu_sisi,rolling_4w_terakhir,p_holm,memburuk,awal_periode_terkini
0,SUP-A,0.034744,0.013875,2.504017,22.481274,0.000000,0.034188,0.000000,1,2026-07-06
1,SUP-B,0.009649,0.013572,0.710918,-1.710788,0.956440,0.009653,1.000000,0,2026-07-06
2,SUP-C,0.036849,0.034718,1.061374,1.748739,0.040168,0.034277,0.160672,0,2026-07-06
3,SUP-D,0.011976,0.012843,0.932489,-1.432087,0.923941,0.009361,1.000000,0,2026-07-06
4,SUP-E,0.010609,0.012187,0.870463,-2.462827,0.993108,0.010642,1.000000,0,2026-07-06


,supplier_code,bulan,bulan_sebelumnya,ng_rate,ng_rate_sebelumnya,selisih,z,p_dua_sisi,p_holm,nyata
0,SUP-A,2026-04,2026-03,0.012482,0.016451,-0.003969,-3.374881,7.384772e-04,0.012554,1
2,SUP-A,2026-06,2026-05,0.011589,0.015218,-0.003629,-2.935552,3.329546e-03,0.046614,1
3,SUP-A,2026-07,2026-06,0.031738,0.011589,0.020149,12.486177,0.000000e+00,0.000000,1
5,SUP-B,2026-04,2026-03,0.009289,0.019024,-0.009735,-3.009239,2.619027e-03,0.039285,1
7,SUP-B,2026-06,2026-05,0.022095,0.011271,0.010824,3.548853,3.869135e-04,0.007351,1
8,SUP-B,2026-07,2026-06,0.009646,0.022095,-0.012448,-3.286233,1.015369e-03,0.016246,1
10,SUP-C,2026-04,2026-03,0.029011,0.034764,-0.005753,-3.611782,3.041002e-04,0.006386,1
11,SUP-C,2026-05,2026-04,0.035478,0.029011,0.006466,3.586833,3.347182e-04,0.006694,1
16,SUP-D,2026-05,2026-04,0.009631,0.014111,-0.004480,-4.740578,2.131091e-06,0.000047,1
17,SUP-D,2026-06,2026-05,0.014932,0.009631,0.005301,5.341570,9.214517e-08,0.000002,1


## Q8 dan Q9: model, stasiun, dan jumlah unit minimum

In [6]:
display(pd.read_csv(R2/'q8_cross_tests.csv')); display(pd.read_csv(R2/'q8_adjusted_or.csv')); display(pd.read_csv(R2/'q9_unit_minimum.csv'))

,supplier_code,dimensi,chi2,derajat_bebas,p_value,ng_rate_tertinggi,ng_rate_terendah
0,SUP-A,model_code,241.349744,2,3.904532e-53,0.028558,0.014063
1,SUP-A,station_code,14.755997,4,5.234994e-03,0.022161,0.017871
2,SUP-B,model_code,22.713112,2,1.169258e-05,0.018296,0.009483
3,SUP-B,station_code,9.269351,4,5.470852e-02,0.016302,0.009861
4,SUP-C,model_code,193.430407,2,9.933921e-43,0.039774,0.019143
5,SUP-C,station_code,11.596268,4,2.062016e-02,0.038032,0.032337
6,SUP-D,model_code,170.359115,2,1.016221e-37,0.017745,0.009344
7,SUP-D,station_code,16.114766,4,2.868969e-03,0.013557,0.010468
8,SUP-E,model_code,243.189782,2,1.556001e-53,0.017636,0.008080
9,SUP-E,station_code,5.573974,4,2.333032e-01,0.013053,0.011247


,supplier_code,or_tanpa_koreksi,or_tanpa_low,or_tanpa_high,or_terkoreksi,or_terkoreksi_low,or_terkoreksi_high,p_terkoreksi,persen_selisih_dijelaskan,ng_rate
0,SUP-A,1.074540,1.071014,1.078078,1.163465,1.160070,1.166869,0,-1.105945,0.019643
1,SUP-B,0.685238,0.679251,0.691279,0.714241,0.708788,0.719735,0,0.109668,0.012929
2,SUP-C,2.556514,2.550972,2.562068,2.053033,2.050143,2.055927,0,0.233663,0.035280
3,SUP-D,0.596845,0.594897,0.598800,0.656345,0.654538,0.658158,0,0.184130,0.012564
4,SUP-E,0.551658,0.549839,0.553484,0.603463,0.601808,0.605122,0,0.150893,0.011813


,acuan,ng_rate,unit_minimum
0,rata-rata semua supplier,0.018572,3300
1,supplier terbaik,0.011813,5200
2,supplier terburuk,0.035280,1800


## Q10 dan Q11: scorecard dan sensitivitas bobot

In [7]:
display(pd.read_csv(R2/'scorecard_latest.csv')); display(pd.read_csv(R2/'sensitivity_grid.csv')); display(pd.read_csv(R2/'red_support.csv'))

,month,supplier_code,units,lots,ng,ng_rate,ci_low,ci_high,lots_eval,lot_cv,trend,s_ng,s_cons,s_trend,score,status,skor_rata_rata_bulan,bulan_merah,bulan_dinilai
0,2026-08,SUP-A,10998,29,376,0.034188,0.030951,0.037750,29,0.904529,0.664172,14.529915,69.849049,33.582825,33.122926,merah,56.621009,3,6
1,2026-08,SUP-B,1036,3,10,0.009653,0.005251,0.017677,3,0.044588,-0.363649,75.868726,98.513746,100.000000,NaN,belum dinilai,83.450307,0,1
2,2026-08,SUP-C,11874,26,407,0.034277,0.031152,0.037703,25,0.853825,-0.110662,14.308573,71.539178,100.000000,50.039081,merah,42.878030,6,6
3,2026-08,SUP-D,20830,42,195,0.009361,0.008141,0.010763,41,0.550461,-0.285544,76.596255,81.651304,100.000000,83.710954,hijau,73.630491,0,6
4,2026-08,SUP-E,18324,37,195,0.010642,0.009255,0.012233,35,0.745014,-0.073276,73.395547,75.166212,100.000000,80.489326,hijau,74.080561,0,6


,sasaran,kombinasi_bobot,urutan_sama_persen,supplier_teratas_sama_persen,supplier_terendah_sama_persen,tau_kendall_min,tau_kendall_rata,status_berubah_SUP-A,status_berubah_SUP-C,status_berubah_SUP-D,status_berubah_SUP-E,status_berubah_SUP-B
0,bulan terakhir,120,1.0,1,1.000,1.0,1.00,0.008333,0.600000,0.025000,0.350000,NaN
1,rata-rata bulan dinilai,120,0.4,1,0.525,0.6,0.87,0.541667,0.491667,0.441667,0.358333,0.033333


,month,supplier_code,score,ng_rate,ng_rate_lainnya,p_unit,p_lot,p_unit_holm,p_lot_holm,didukung
0,2026-03,SUP-A,58.291205,0.016451,0.018189,0.953353,9.042439e-01,0.953353,9.042439e-01,0
1,2026-03,SUP-C,32.698730,0.034764,0.012938,0.000000,2.557989e-19,0.000000,2.046391e-18,1
2,2026-04,SUP-C,56.334760,0.029011,0.013253,0.000000,2.690909e-20,0.000000,2.421818e-19,1
3,2026-05,SUP-C,43.442012,0.035478,0.011667,0.000000,3.613869e-14,0.000000,2.168321e-13,1
4,2026-06,SUP-C,35.372692,0.040286,0.013158,0.000000,2.191026e-16,0.000000,1.533718e-15,1
5,2026-07,SUP-A,25.314697,0.031738,0.020618,0.000000,6.543893e-04,0.000000,1.308779e-03,1
6,2026-07,SUP-C,39.380904,0.038938,0.018542,0.000000,1.206273e-13,0.000000,6.031366e-13,1
7,2026-08,SUP-A,33.122926,0.034188,0.015500,0.000000,3.085562e-08,0.000000,1.234225e-07,1
8,2026-08,SUP-C,50.039081,0.034277,0.015160,0.000000,3.454234e-08,0.000000,1.234225e-07,1


## Q12: dampak (estimated)

In [8]:
display(pd.read_csv(R2/'impact_scenarios.csv'))

,skenario,supplier,ng_rate_supplier,ng_rate_pembanding,selisih,volume_bulanan,ng_terhindari_per_bulan,ci_low,ci_high,persen_ng_supplier_terhindari,label
0,"dasar: pembanding gabungan, volume 4 bulan penuh",SUP-C,0.035280,0.014103,0.021177,20957.75,443.820048,349.283369,545.685656,0.600256,estimated
1,sensitivitas: pembanding rata-rata sederhana s...,SUP-C,0.035280,0.014237,0.021043,20957.75,441.005411,NaN,NaN,0.596449,estimated
2,sensitivitas: volume rata-rata semua bulan (Ag...,SUP-C,0.035280,0.014103,0.021177,19685.50,416.877745,NaN,NaN,0.600256,estimated
3,tambahan: SUP-A kembali ke NG rate sebelum mem...,SUP-A,0.034744,0.013875,0.020869,20050.50,418.428149,NaN,NaN,0.600642,estimated


## Q13 dan Q14: kualitas data dan cek silang

In [9]:
display(pd.read_csv(R1/'data_quality.csv')); display(pd.read_csv(R2/'crosscheck_sql_pandas.csv')); display(pd.read_csv(R2/'crosscheck_excel.csv'))

,kategori,jumlah,dasar,proporsi,keputusan
0,baris lot (mentah),1200,1200,1.000000,dasar
1,baris pemakaian (mentah),3584,3584,1.000000,dasar
2,lot tanpa supplier atau material,0,1200,0.000000,tidak ada
3,lot ganda,0,1200,0.000000,tidak ada
4,pasangan supplier-material tidak ada di dim_su...,0,1200,0.000000,tidak ada
5,baris pemakaian ganda,0,3584,0.000000,tidak ada
6,pemakaian tanpa lot,0,3584,0.000000,tidak ada
7,NG lebih besar dari input,0,3584,0.000000,tidak ada
8,input tidak positif,0,3584,0.000000,tidak ada
9,pemakaian sebelum lot diterima,0,3584,0.000000,tidak ada


,pemeriksaan,dibandingkan,cocok,selisih_maks,hasil
0,NG rate per supplier (Q1),5,5,4.700000e-11,cocok
1,NG rate per supplier x material (Q1),7,7,3.500000e-11,cocok
2,"NG rate bulanan, RANK, DENSE_RANK, LAG (Q3)",30,30,5.000000e-11,cocok
3,rolling NG rate 4 minggu dan LAG (Q6),130,130,5.000000e-11,cocok
4,NG rate per lot (Q4),1200,1200,5.000000e-11,cocok
5,koefisien variasi antar lot (Q4),5,5,5.000000e-11,cocok
6,komponen scorecard: CV antar lot dan tren (Q10),30,30,5.000000e-11,cocok


,month,supplier_code,units_excel,units_sql,ng_excel,ng_sql,rate_excel,rate_sql,rank_excel,rank_sql,cocok
0,2026-06,SUP-A,14756,14756,171,171,0.011589,0.011589,4,4,1
1,2026-06,SUP-B,3485,3485,77,77,0.022095,0.022095,2,2,1
2,2026-06,SUP-C,20255,20255,816,816,0.040286,0.040286,1,1,1
3,2026-06,SUP-D,23908,23908,357,357,0.014932,0.014932,3,3,1
4,2026-06,SUP-E,27240,27240,308,308,0.011307,0.011307,5,5,1
5,2026-08,SUP-A,10998,10998,376,376,0.034188,0.034188,2,2,1
6,2026-08,SUP-B,1036,1036,10,10,0.009653,0.009653,4,4,1
7,2026-08,SUP-C,11874,11874,407,407,0.034277,0.034277,1,1,1
8,2026-08,SUP-D,20830,20830,195,195,0.009362,0.009362,5,5,1
9,2026-08,SUP-E,18324,18324,195,195,0.010642,0.010642,3,3,1


## Hipotesis dan uji pemulihan ground truth

In [10]:
display(pd.read_csv(R2/'hypotheses.csv')[['id','verdict','bukti']]); display(pd.read_csv(R2/'ground_truth_check.csv')[['id','fakta','hasil']])

,id,verdict,bukti
0,H1,Terbukti,material dengan supplier tertinggi signifikan ...
1,H2,Tidak terbukti,0 dari 5 supplier punya lebih dari 50% NG dari...
2,H3,Terbukti,supplier memburuk (8 minggu terakhir vs sebelu...
3,H4,Terbukti,supplier terburuk SUP-C: OR 2.56 menjadi 2.05;...
4,H5,Terbukti,supplier volume terkecil SUP-B: CI bulanan ter...


,id,fakta,hasil
0,1,Supplier dengan NG rate tertinggi,Sesuai
1,2,Sebagian kecil lot menyumbang banyak NG,Sesuai
2,3,Supplier memburuk sejak hari ke-90 (+-2 bulan ...,Sesuai
3,4,Perbedaan sebagian karena model: supplier terb...,Sebagian
4,5,Supplier volume kecil belum bisa diberi status,Sesuai
